# FashionMNIST 分类（PyTorch 版本）

本 Notebook 使用 PyTorch 完成与原 MindSpore 文件相同的任务：
- 数据加载与可视化
- CNN 模型构建
- 训练与测试
- 模型保存与加载
- 预测结果展示

In [ ]:
# 导入依赖库
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 固定随机种子，保证结果更可复现
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# 数据准备
BATCH_SIZE = 64
DATA_ROOT = './data'

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root=DATA_ROOT, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print('Train size:', len(train_dataset))
print('Test size:', len(test_dataset))

In [ ]:
# 显示前 10 张训练图片及标签
images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    img = images[i].squeeze(0).numpy()
    plt.imshow(img, cmap='gray')
    plt.title(class_names[labels[i].item()], fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# 模型构建
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = Network().to(device)
print(model)

In [ ]:
# 损失函数与优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-2)

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if batch_idx % 100 == 0:
            print(f'Batch [{batch_idx:3d}/{len(dataloader):3d}]  Loss: {loss.item():.4f}')

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
# 训练与测试
EPOCHS = 10
history = {
    'train_loss': [], 'train_acc': [],
    'test_loss': [], 'test_acc': []
}

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch + 1}/{EPOCHS}')
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}%')
    print(f'Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc * 100:.2f}%')

print('训练完成')

In [ ]:
# 可视化训练曲线
epochs_axis = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_axis, history['train_loss'], label='Train Loss')
plt.plot(epochs_axis, history['test_loss'], label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_axis, history['train_acc'], label='Train Acc')
plt.plot(epochs_axis, history['test_acc'], label='Test Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 保存模型参数
save_path = 'clothes_model_pytorch.pth'
torch.save(model.state_dict(), save_path)
print(f'Saved model to {save_path}')

In [ ]:
# 加载模型并进行预测展示
loaded_model = Network().to(device)
loaded_model.load_state_dict(torch.load('clothes_model_pytorch.pth', map_location=device))
loaded_model.eval()

images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = loaded_model(images)
    preds = outputs.argmax(dim=1)

images_np = images.cpu().numpy()
labels_np = labels.cpu().numpy()
preds_np = preds.cpu().numpy()

plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images_np[i].squeeze(0), cmap='gray')
    p_name = class_names[preds_np[i]]
    a_name = class_names[labels_np[i]]
    color = 'green' if preds_np[i] == labels_np[i] else 'red'
    plt.title(f'P: {p_name}\nA: {a_name}', fontsize=9, color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()